# Анализ тем

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

topics = [
    "Алкоголь, наркотики и зависимость",
    "Финансы",
    "Животные и домашние питомцы",
    "Здоровье и медицина",
    "Интернет и технологии",
    "Истории и байки",
    "Кино и музыка",
    "Национальности и стереотипы",
    "Отношения",
    "Политика и общество",
    "Преступность, тюрьма и правосудие",
    "Психотерапия",
    "Работа и учеба",
    "Секс и половые связи",
    "Семья, дети и родственники",
    "Спорт",
    "Смерть",
    "Философия и смысл жизни",
    "Эмиграция, переезд",
    "Мода и внешний вид",
]

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
embeddings = model.encode(topics)

# Расчет косинусного сходства
similarity_matrix = cosine_similarity(embeddings)

# Визуализация матрицы сходства
plt.figure(figsize=(12, 10))
sns.heatmap(
    similarity_matrix,
    xticklabels=topics,
    yticklabels=topics,
    cmap="coolwarm",
    annot=False,
)
plt.title("Косинусная близость между темами")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


# TEST

### Получение метаданных плейлиста

In [1]:
import youtube_downloader

playlist_info = youtube_downloader.yt_playlist_extract_info(
    "https://www.youtube.com/watch?v=MaVc3dqiEI4&list=PLcQngyvNgfmLi9eyV9reNMqu-pbdKErKr"
)

### Загрузка метаданных плейлиста в Postgres

In [9]:
import re


def normalize_title(title: str) -> str:
    # Заменяем всё, что не буква или цифра, на подчёркивание
    temp = re.sub(r"[^\w\dа-яА-ЯёЁ]+", "_", title)
    # Удаляем повторяющиеся подчёркивания
    temp = re.sub(r"__+", "_", temp)
    # Удаляем подчёркивания в начале и в конце
    return temp.strip("_")


# Пример
s = "Вечер Судного Дня. ___1!!Дело： еда против комфорта"
print(normalize_title(s))


Вечер_Судного_Дня_1_Дело_еда_против_комфорта


In [4]:
import youtube_downloader as yd
import config
import yt_dlp

video_url = "https://www.youtube.com/watch?v=duXr-dUXtw4"

with yt_dlp.YoutubeDL(config.YDL_OPTS) as ydl:
    # 1. Получаем информацию о видео, не скачивая его
    info_dict = ydl.extract_info(video_url, download=False)

    # 2. Определяем ожидаемый путь к файлу
    filename = ydl.prepare_filename(info_dict)
    # audio_path = Path(filename).with_suffix(".m4a")

    # # 3. Проверяем, существует ли файл
    # if audio_path.exists():
    #     logging.info(f"Файл скачан ранее: {audio_path}")
    #     return audio_path

    # # 4. Если файла нет, запускаем полное скачивание и обработку
    # ydl.extract_info(video_url, download=True)

    # logging.info("Скачивание аудио завершено")

filename

'/Users/aleksandr/Documents/Projects/StandUP_project/data/audio/Вечер Судного Дня. Дело： еда против комфорта.m4a'

In [11]:
import psycopg

# Параметры подключения (замените на свои)
conn_params = {
    "dbname": "standup",
    "user": "standup",
    "password": "standup",
    "host": "postgresql.standup.orb.local",
    "port": "5432",
}

try:
    # Подключение к БД
    conn = psycopg.connect(**conn_params)
    cur = conn.cursor()

    # SQL-запрос для вставки
    insert_query = """
    INSERT INTO standup_raw.process_video (
        channel_id, channel_name, playlist_id, playlist_title, video_id, video_title, 
        viedeo_url, duration, view_count, comment_count, like_count, upload_date, 
        transcribe_json, llm_chapter_json, SoundFileClassifier_app
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    # Подготовка данных: список кортежей из значений словарей
    data = [
        (
            item["channel_id"],
            item["channel_name"],
            item["playlist_id"],
            item["playlist_title"],
            item["video_id"],
            item["video_title"],
            item["viedeo_url"],
            item["duration"],
            item["view_count"],
            item["comment_count"],
            item["like_count"],
            item["upload_date"],
            item["transcribe_json"],
            item["llm_chapter_json"],
            item["SoundFileClassifier_app"],
        )
        for item in playlist_info
    ]

    # Выполнение множественной вставки
    cur.executemany(insert_query, data)

    # Коммит изменений
    conn.commit()
    print(f"Успешно вставлено {len(playlist_info)} записей.")

except (Exception, psycopg.Error) as error:
    print(f"Ошибка: {error}")
finally:
    if cur:
        cur.close()
    if conn:
        conn.close()


Успешно вставлено 5 записей.


### Наладка цикла загрузки всех данных через функции и загрузка в Postgres

## Итерация по метаданным и запись в json ответы

In [1]:
import time
from io import StringIO
import pandas as pd
from google import genai
from google.genai import types
import config
import requests
import re
import json
import yt_dlp

TIME_RE = re.compile(
    r"(\d{2}):(\d{2}):(\d{2}),(\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2}),(\d{3})"
)


def yt_extract_info(video_url):
    with yt_dlp.YoutubeDL(config.YDL_OPTS) as ydl:
        info_dict = ydl.extract_info(video_url, download=False)
        return info_dict


def to_seconds_int(h, m, s, ms):
    """
    Переводит компоненты таймкода в секунды,
    округляя до целого числа.
    """
    total = int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000
    return int(round(total))


def format_srt(raw_srt: str, as_json: bool = False):
    lines = raw_srt.splitlines()
    items = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if re.fullmatch(r"\d+", line):
            i += 1
            continue
        m = TIME_RE.match(line)
        if not m:
            i += 1
            continue
        start = to_seconds_int(*m.groups()[:4])
        end = to_seconds_int(*m.groups()[4:])
        i += 1
        txt_lines = []
        while i < len(lines) and lines[i].strip():
            txt_lines.append(lines[i].strip())
            i += 1
        txt = " ".join(txt_lines)
        txt = re.sub(r"\[|\]|>>", "", txt).strip().lower()
        items.append((start, end, txt))

    if as_json:
        return json.dumps(
            [{"start": s, "end": e, "text": t} for s, e, t in items],
            ensure_ascii=False,
            indent=2,
        )
    else:
        # Только стартовый таймкод и текст
        return "\n".join(f"{s} {t}" for s, _, t in items)


df_full = pd.read_json("../data/test/df_full.json")
df_full = df_full.iloc[140:141]

for index, row in df_full.iterrows():
    try:
        title = row["title"]
        url = row["original_url"]
        duration = row["duration"]

        info_dict = yt_extract_info(url)
        captions = info_dict.get("automatic_captions", {}).get("ru", [])
        srt_entry = next((item for item in captions if item.get("ext") == "srt"), None)
        srt_url = srt_entry["url"]

        # !!! ошибка 'NoneType' object is not subscriptable - обработать ошибку отсутствия сабов !!! нужно будет использовать транскрибацию
        response = requests.get(srt_url)

        if response.status_code == 200:
            srt_text = response.text
        else:
            raise Exception(f"Ошибка загрузки: {response.status_code}")

        tsv_content = format_srt(srt_text, as_json=False)

        client = genai.Client(
            api_key=config.GEMINI_API_KEY,
        )

        model = "gemini-2.5-flash"

        promt = f"""
        **Роль:** Ты — продвинутый ИИ-аналитик, эксперт в тематическом анализе юмористических шоу и стендапов.
        **Задача:** Проанализировать предоставленный транскрипт выступления и выделить основные смысловые темы, указав их таймкоды.

        ВАЖНАЯ ИНФОРМАЦИЯ:
        Общая длительность выступления: {duration} секунд.

        # Правила Анализа
        1.  **Крупные блоки:** Выделяй только крупные, ключевые темы. Тема — это смысловой блок, обсуждаемый несколько минут (например, "отношения с родителями", "опыт жизни за границей", "неловкие ситуации"). Не дроби выступление на отдельные шутки.
        2.  **Строгая хронология:** Темы должны идти в том же порядке, в котором они появляются. Массив `timestamp` должен быть строго отсортирован по возрастанию.
        3.  **Рекламные интеграции:** Обязательно распознавай рекламные блоки. Для них используй строго только название темы (theme) "Реклама".
        4.  **Контроль длительности:** Последний таймкод НЕ ДОЛЖЕН превышать общую длительность выступления ({duration} секунд).
        5.  **Контроль пропорций:** Ни одна тема не должна занимать более трети от общего времени выступления.
        """

        generate_content_config = types.GenerateContentConfig(
            temperature=0,
            thinking_config=types.ThinkingConfig(
                thinking_budget=-1,
            ),
            response_mime_type="application/json",
            response_schema=genai.types.Schema(
                type=genai.types.Type.OBJECT,
                required=["theme", "timestamp"],
                properties={
                    "theme": genai.types.Schema(
                        type=genai.types.Type.ARRAY,
                        items=genai.types.Schema(
                            type=genai.types.Type.STRING,
                        ),
                    ),
                    "timestamp": genai.types.Schema(
                        type=genai.types.Type.ARRAY,
                        items=genai.types.Schema(
                            type=genai.types.Type.INTEGER,
                        ),
                    ),
                },
            ),
            system_instruction=[
                types.Part.from_text(text=promt),
            ],
        )

        response = client.models.generate_content(
            model=model,
            contents=types.Content(
                parts=[
                    types.Part(text=tsv_content),
                ]
            ),
            config=generate_content_config,
        )
        # Сохранение результата
        # !!! All arrays must be of the same length !!! обработать ошибку некорректной выдачи от LLM, списки разной длины и фейлится созранение
        df_chunk = pd.read_json(StringIO(response.text))
        out_path = f"../data/test/time_text/{title}.json"
        df_chunk.to_json(out_path, force_ascii=False, indent=2)
        print(title, " - обработано")
    except Exception as e:
        print(title, f" - ошибка {e}")
    time.sleep(10)

Разгоны #18 [Александр Киселёв, Дима Гаврилов, Давид Квахаджелидзе, Коля Андреев,  Эльдар Гусейнов]  - ошибка Ошибка загрузки: 429


KeyboardInterrupt: 

In [4]:
import pandas as pd

df_full = pd.read_json("../data/test/df_full.json")
df_full = df_full.iloc[140:]
df_full

,id,title,duration,view_count,comment_count,like_count,upload_date,chapters,original_url
140,2NzW8EkZ9Hc,"Разгоны #18 [Александр Киселёв, Дима Гаврилов,...",3472,1342493,1000,30869,20200828,None,https://www.youtube.com/watch?v=2NzW8EkZ9Hc
141,ivmQ3tM12dY,"Разгоны #17 [Идрак Мирзализаде, Эльдар Гусейно...",3915,2105354,1200,43248,20200814,None,https://www.youtube.com/watch?v=ivmQ3tM12dY
142,TVtakPe6kL4,"Разгоны #16 [новогодние праздники, откровения ...",2380,1050760,978,37966,20200620,None,https://www.youtube.com/watch?v=TVtakPe6kL4
143,rWv9HTi6Kh4,"Разгоны #15 [эволюция, пердёж, суперклей]",3133,1517821,989,35105,20200503,None,https://www.youtube.com/watch?v=rWv9HTi6Kh4
144,uvZ2vKQUgZ4,Разгоны #14 [Анна Карёнина/ ремонт/ красота по...,1793,998212,604,26169,20200320,None,https://www.youtube.com/watch?v=uvZ2vKQUgZ4
145,xw0mX3XTVWg,Разгоны #13 [продуктовые/ гляделки/ криокамеры],1581,1260336,1000,34445,20200124,None,https://www.youtube.com/watch?v=xw0mX3XTVWg
146,0Tbvlh5a6Wc,Разгоны #12 [бандана/ халва/ чучела],3228,1857701,1300,40898,20191220,None,https://www.youtube.com/watch?v=0Tbvlh5a6Wc
147,u0Ury_yZW5M,Разгоны #11 [акинатор/ утро с девушкой/ цитаты...,2715,1323987,813,29780,20191113,None,https://www.youtube.com/watch?v=u0Ury_yZW5M
148,xfLIlmM59MA,Разгоны #10 [шарики/лунатизм/имена],3890,3639855,2000,61745,20191005,None,https://www.youtube.com/watch?v=xfLIlmM59MA
149,-V-o5m6rIcw,Разгоны #9 [коты/роботы/самолеты],2833,1545736,1300,38593,20190915,None,https://www.youtube.com/watch?v=-V-o5m6rIcw


# Наладка отдельных функций

## Наладка транскрибации

In [ ]:
import transcribe

result = transcribe.transcribe_audio(
    "../data/audio/Вася Медведев - Встреча с богами [Шоу Историй].m4a"
)

'{\n  "id":{\n    "0":0,\n    "1":1,\n    "2":2,\n    "3":3,\n    "4":4,\n    "5":5,\n    "6":6,\n    "7":7,\n    "8":8,\n    "9":9,\n    "10":10,\n    "11":11,\n    "12":12,\n    "13":13,\n    "14":14,\n    "15":15,\n    "16":16,\n    "17":17,\n    "18":18,\n    "19":19,\n    "20":20,\n    "21":21,\n    "22":22,\n    "23":23,\n    "24":24,\n    "25":25,\n    "26":26,\n    "27":27,\n    "28":28,\n    "29":29,\n    "30":30,\n    "31":31,\n    "32":32,\n    "33":33,\n    "34":34,\n    "35":35,\n    "36":36,\n    "37":37,\n    "38":38,\n    "39":39,\n    "40":40,\n    "41":41,\n    "42":42,\n    "43":44,\n    "44":45,\n    "45":46,\n    "46":47,\n    "47":48,\n    "48":49,\n    "49":50,\n    "50":51,\n    "51":52,\n    "52":53,\n    "53":54,\n    "54":55,\n    "55":56,\n    "56":57,\n    "57":58,\n    "58":59,\n    "59":60,\n    "60":61,\n    "61":62,\n    "62":63,\n    "63":64,\n    "64":65,\n    "65":66,\n    "66":67,\n    "67":68,\n    "68":69,\n    "69":70,\n    "70":71,\n    "71":72,

In [5]:
import pandas as pd
from io import StringIO

df = pd.read_json(StringIO(result))
df

,id,seek,start,end,text,tokens,temperature,avg_logprob,compression_ratio,no_speech_prob,words,next_start
0,0,0,0,30,Это жёлая история Я сегодня хотел с вами поде...,"[50365, 6684, 2989, 24180, 4251, 41531, 13, 50...",0,-0.653411,0.825000,0,NaN,30.0
1,1,3000,30,32,В котором я давно не виделся.,"[50365, 2348, 39818, 2552, 40086, 1725, 40718,...",0,-0.240090,1.821277,0,NaN,2.0
2,2,3000,32,36,"И мы с ним встретились, пили вино на лестничн...","[50465, 3272, 4777, 776, 25793, 47647, 19285, ...",0,-0.240090,1.821277,0,NaN,4.0
3,3,3000,36,41,"Мы были студентами, отрывались по полной.","[50665, 12726, 14355, 44322, 5243, 5150, 11, 2...",0,-0.240090,1.821277,0,NaN,5.0
4,4,3000,41,50,"Всё было круто, пока в один момент он не созв...","[50915, 29661, 8060, 26970, 860, 11, 17770, 74...",0,-0.240090,1.821277,0,NaN,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...
80,81,29300,294,297,"Типа, давайте все сделаем одну и ту же татуир...","[50415, 3200, 13306, 386, 11, 30412, 4640, 103...",0,-0.138512,1.846154,0,NaN,4.0
81,82,29300,298,300,Давайте все сделаем одни и те же шрамы на руках.,"[50615, 30487, 4640, 10326, 7906, 5693, 1903, ...",0,-0.138512,1.846154,0,NaN,3.0
82,83,29300,301,302,"Мы решили сделать что-то подобное,","[50765, 12726, 14025, 5435, 18695, 2143, 12, 8...",0,-0.138512,1.846154,0,NaN,2.0
83,84,29300,303,304,только мы начали...,"[50865, 9008, 4777, 8970, 4071, 485, 50915]",0,-0.138512,1.846154,0,NaN,2.0


In [4]:
import pandas as pd
from io import StringIO

df = pd.read_json(StringIO(result))

# Округляем колонку 'start' до целого
df["start"] = df["start"].round().astype(int)

# Выбираем только нужные колонки
df_export = df[["start", "text"]]

# Сохраняем в TSV-файл
text = df_export.to_csv(sep="\t", index=False)
text

'start\ttext\n0\t Это жёлая история Я сегодня хотел с вами поделиться самой странной историей, наверное, в моей жизни Не знаю, может у вас тоже бывали такие ситуации, в которые вы никогда не планировали бы попасть Но жизнь постепенно погружает вас в эту ситуацию Понимаете, о чём я говорю? Короче, ладно, давайте я расскажу историю Я в тот вечер собрался встретиться со своим одноклассником, которым...\n30\t В котором я давно не виделся.\n32\t И мы с ним встретились, пили вино на лестничной клетке.\n36\t Мы были студентами, отрывались по полной.\n41\t Всё было круто, пока в один момент он не созвонился с кем-то и не сказал мне, что сейчас придут боги.\n52\t Боги.\n54\t Боги.\n56\t Я как бы православный человек, понимаете?\n60\t И мне этот момент очень заинтересовал.\n63\t И я остался ждать, естественно, богов, пока они придут.\n68\t Приезжают боги, значит, на лифте.\n75\t Ну, я думаю, кнопки они не нажимали, они же боги всё-таки.\n79\t Я с ними знакомлюсь.\n81\t Ну, у них имена не типа та